# ZynNova：ZynMorph 电极微结构模块完整测试

本 notebook 直接调用 `zynnova.zynmorph`，覆盖以下能力：

1. 正极或负极三相微结构的条件化生成；
2. 相体积分数的**精确体素配额**；
3. 指定相沿指定轴的贯通约束；
4. 连通率、比表面积、相间界面面积等形貌指标；
5. NPZ、NPY、RAW、TIFF 体素数据往返验证；
6. VTK、Gmsh MSH、Abaqus INP 和边界 PLY/STL 导出；
7. 每个体素拆分为 6 个 Tet4 后的有限元质量门禁；
8. 随机种子可复现性、相关长度条件扫描；
9. 从若干二维标签切片重建三维体素场；
10. 生成机器可读的最终测试报告。

> **坐标约定**：体素数组使用 `labels[z, y, x]`；有限元节点坐标保存为 `(x, y, z)`。
>
> **规模约定**：每个体素生成 6 个四面体。默认使用中等规模快速测试；将 `FAST_MODE=False` 后可测试更大的电极区域，但导出的 ASCII 网格会显著增大。


## 0. 安装与运行位置

推荐在 ZynNova 仓库根目录运行：

```bash
python -m pip install -e ".[zynnova]"
python -m pip install pandas matplotlib
jupyter lab
```

若已经以 editable 或 wheel 方式安装 `zynnova`，本 notebook 可以放在任意目录。若尚未安装，它会从当前目录及其父目录中自动寻找 `src/zynnova`。也可以通过环境变量指定仓库：

```bash
export ZYNNOVA_ROOT=/path/to/ZynNova
```


In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
import time
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def _find_zynnova_root() -> Path | None:
    explicit = os.environ.get("ZYNNOVA_ROOT")
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "src" / "zynnova").is_dir():
            return candidate
    return None


try:
    import zynnova
except ModuleNotFoundError:
    PROJECT_ROOT = _find_zynnova_root()
    if PROJECT_ROOT is None:
        raise RuntimeError(
            "未找到 zynnova。请先执行 `python -m pip install -e .`，"
            "或设置环境变量 ZYNNOVA_ROOT 指向 ZynNova 仓库。"
        )
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
    import zynnova
else:
    PROJECT_ROOT = _find_zynnova_root()

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("ZynNova:", zynnova.__version__)
print("Project root:", PROJECT_ROOT if PROJECT_ROOT is not None else "installed package")


In [ ]:
from zynnova.geometry import (
    tetrahedron_mean_ratio,
    tetrahedron_signed_volumes,
)
from zynnova.zynmorph import (
    BACKENDS,
    BatteryPhase,
    GenerationConfig,
    MicrostructureCondition,
    MicrostructureVolume,
    SliceObservation,
    SpectralConditionalGenerator,
    analyze_microstructure,
    reconstruct_from_slices,
    run_zynmorph,
)

try:
    from PIL import Image, ImageSequence
except ImportError:
    Image = None
    ImageSequence = None

backend_df = pd.DataFrame(BACKENDS.status())
display(backend_df)

spectral_status = backend_df.loc[backend_df["name"] == "spectral-exact"].iloc[0]
assert bool(spectral_status["available"]), spectral_status.to_dict()
print("spectral-exact 后端可用。")


## 1. 当前实现边界（测试时必须明确）

`MicrostructureCondition` 中不同字段在当前内置后端中的作用并不完全相同：

- `shape`、`phase_fractions`、`correlation_lengths_voxels`、`interface_affinity`、`seed` 和 `temperature` 直接参与 `spectral-exact` 生成；
- `percolation_axes` 在生成后通过保持相数量不变的标签交换强制执行；
- `descriptor_targets` 通过随机跨相标签交换进行局部细化，目标键必须来自 `metrics.flatten()`；
- `voxel_size_m` 决定物理尺寸、界面面积和有限元节点坐标；
- `manufacturing` 会被完整记录，并可作为训练型 `torch-rectified-flow` 检查点的条件向量，但当前 `spectral-exact` 不会主动解释压实率等工艺变量；
- 当前 `spectral-exact` 并未读取 `periodic` 布尔值。谱随机场由 FFT 构造，而连通性与界面统计仍按非周期外边界计算。

因此，本 notebook 对已经真实实现的约束做硬断言，对尚未由本地谱后端消费的工艺字段只检查序列化是否保留，不伪称其已经产生工艺因果响应。


In [ ]:
phase_schema = pd.DataFrame(
    [
        {
            "phase_id": int(phase),
            "enum_name": phase.name,
            "canonical_name": phase.name.lower(),
        }
        for phase in BatteryPhase
    ]
)
display(phase_schema)


## 2. 测试参数

将 `ELECTRODE_KIND` 改为 `"negative"` 即可用负极活性相、电解液相和负极 CBD 标签执行同一套测试。


In [ ]:
# ------------------------- 用户可修改区域 -------------------------
ELECTRODE_KIND = "positive"       # "positive" 或 "negative"
FAST_MODE = True                  # False: 使用更大的默认体素网格
RUN_PARAMETER_SWEEP = True
RUN_SLICE_RECONSTRUCTION = True
SEED = 23

SHAPE = (18, 32, 32) if FAST_MODE else (48, 64, 64)  # (z, y, x)
VOXEL_SIZE_M = (100e-9, 100e-9, 100e-9)              # (dz, dy, dx)
OUTPUT_ROOT = Path.cwd() / "zynnova_runs" / "electrode_microstructure_test"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# ------------------------------------------------------------------

if ELECTRODE_KIND == "positive":
    ACTIVE_PHASE = int(BatteryPhase.POSITIVE_ACTIVE)
    ELECTROLYTE_PHASE = int(BatteryPhase.POSITIVE_ELECTROLYTE)
    CBD_PHASE = int(BatteryPhase.POSITIVE_CBD)
elif ELECTRODE_KIND == "negative":
    ACTIVE_PHASE = int(BatteryPhase.NEGATIVE_ACTIVE)
    ELECTROLYTE_PHASE = int(BatteryPhase.NEGATIVE_ELECTROLYTE)
    CBD_PHASE = int(BatteryPhase.NEGATIVE_CBD)
else:
    raise ValueError("ELECTRODE_KIND 必须为 'positive' 或 'negative'")

n_voxels = int(np.prod(SHAPE))
n_tetrahedra = 6 * n_voxels
n_nodes = int(np.prod(np.asarray(SHAPE) + 1))
estimated_core_bytes = n_nodes * 3 * 8 + n_tetrahedra * 4 * 8 + n_tetrahedra * 4

size_table = pd.DataFrame(
    [
        {"item": "voxel shape (z,y,x)", "value": str(SHAPE)},
        {"item": "voxels", "value": f"{n_voxels:,}"},
        {"item": "Tet4 cells", "value": f"{n_tetrahedra:,}"},
        {"item": "structured nodes", "value": f"{n_nodes:,}"},
        {"item": "core mesh arrays estimate", "value": f"{estimated_core_bytes / 2**20:.1f} MiB"},
        {"item": "physical size (µm, z-y-x)", "value": str(tuple(round(SHAPE[i] * VOXEL_SIZE_M[i] * 1e6, 4) for i in range(3)))},
        {"item": "output root", "value": str(OUTPUT_ROOT.resolve())},
    ]
)
display(size_table)


In [ ]:
condition = MicrostructureCondition(
    shape=SHAPE,
    phase_fractions={
        ACTIVE_PHASE: 0.58,
        ELECTROLYTE_PHASE: 0.28,
        CBD_PHASE: 0.14,
    },
    voxel_size_m=VOXEL_SIZE_M,
    correlation_lengths_voxels={
        ACTIVE_PHASE: (3.2, 4.5, 4.5),
        ELECTROLYTE_PHASE: (2.4, 3.0, 3.0),
        CBD_PHASE: (1.4, 2.0, 2.0),
    },
    interface_affinity={
        (ACTIVE_PHASE, CBD_PHASE): 0.55,
        (ACTIVE_PHASE, ELECTROLYTE_PHASE): -0.10,
        (ELECTROLYTE_PHASE, CBD_PHASE): 0.15,
    },
    percolation_axes={
        ACTIVE_PHASE: (0, 1, 2),
        ELECTROLYTE_PHASE: (0,),
        CBD_PHASE: (2,),
    },
    manufacturing={
        "calendering_ratio": 0.18,
        "binder_fraction": 0.14,
    },
    descriptor_targets={
        f"phase_{ELECTROLYTE_PHASE}.connected_fraction": 0.85,
    },
    periodic=(False, True, True),
    seed=SEED,
)

volume_formats = ("npz", "npy", "raw", "tiff") if Image is not None else ("npz", "npy", "raw")
config = GenerationConfig(
    backend="spectral-exact",
    refinement_steps=6 if FAST_MODE else 16,
    temperature=0.15,
    preserve_exact_fractions=True,
    output_directory=str(OUTPUT_ROOT),
    export_volume_formats=volume_formats,
    export_mesh_formats=("vtk", "msh", "inp"),
    maximum_tetrahedra=max(2_000_000, n_tetrahedra + 6),
)

condition_table = pd.DataFrame(
    [
        {
            "phase_id": phase,
            "phase": BatteryPhase(phase).name,
            "target_fraction": condition.phase_fractions[phase],
            "exact_voxels": condition.exact_phase_counts()[phase],
            "correlation_zyx": condition.correlation_lengths_voxels.get(phase),
            "required_percolation_axes": condition.percolation_axes.get(phase, ()),
        }
        for phase in condition.phases
    ]
)
display(condition_table)


## 3. 执行完整生成、分析、网格化与导出流水线

这个单元格会创建唯一的运行目录，不会覆盖之前的结果。


In [ ]:
started = time.perf_counter()
result = run_zynmorph(condition, config)
elapsed_seconds = time.perf_counter() - started

labels = result.generation.volume.labels
print(f"完成，用时 {elapsed_seconds:.3f} s")
print("Run directory:", result.directory)
print("Backend:", result.generation.backend)
print("Shape:", labels.shape)
print("Physical size (m, z-y-x):", result.generation.volume.physical_size_m)
print("Refinement loss:", result.generation.refinement_loss)
print("Percolation repairs:", result.generation.metadata.get("percolation_repairs", []))


## 4. 相组成与形貌统计


In [ ]:
phase_rows = []
for phase in condition.phases:
    metric = result.metrics.phases[phase]
    target_count = condition.exact_phase_counts()[phase]
    observed_count = int(np.count_nonzero(labels == phase))
    phase_rows.append(
        {
            "phase_id": phase,
            "phase": BatteryPhase(phase).name,
            "target_voxels": target_count,
            "observed_voxels": observed_count,
            "target_fraction": condition.phase_fractions[phase],
            "observed_fraction": metric.volume_fraction,
            "fraction_error": metric.volume_fraction - condition.phase_fractions[phase],
            "connected_fraction": metric.connected_fraction,
            "percolates_z": metric.percolates[0],
            "percolates_y": metric.percolates[1],
            "percolates_x": metric.percolates[2],
            "specific_surface_1_per_um": metric.specific_surface_area_per_m / 1e6,
        }
    )
phase_metrics_df = pd.DataFrame(phase_rows)
display(phase_metrics_df)

interface_rows = []
for pair, area_m2 in sorted(result.metrics.interface_area_m2.items()):
    interface_rows.append(
        {
            "phase_a": pair[0],
            "name_a": BatteryPhase(pair[0]).name,
            "phase_b": pair[1],
            "name_b": BatteryPhase(pair[1]).name,
            "area_m2": float(area_m2),
            "area_um2": float(area_m2) * 1e12,
        }
    )
interface_df = pd.DataFrame(interface_rows)
display(interface_df)


## 5. 必须通过的硬门禁

下面的断言任意一项失败，都说明生成、导出或有限元网格存在实际问题，而不是只看图“似乎正常”。


In [ ]:
# 5.1 基本体素与相标签
assert labels.dtype == np.int32
assert labels.shape == condition.shape
assert set(int(v) for v in np.unique(labels)) == set(condition.phases)

# 5.2 精确相配额
observed_counts = {
    phase: int(np.count_nonzero(labels == phase)) for phase in condition.phases
}
expected_counts = dict(condition.exact_phase_counts())
assert observed_counts == expected_counts, (observed_counts, expected_counts)
assert dict(result.generation.achieved_counts) == expected_counts
assert dict(result.generation.exact_counts) == expected_counts

# 5.3 指定相贯通
for phase, axes in condition.percolation_axes.items():
    observed = result.metrics.phases[phase].percolates
    for axis in axes:
        assert observed[axis], (phase, axis, observed)

# 5.4 Tet4 数量与材料域映射
mesh = result.fem.mesh
assert mesh.n_cells == labels.size * 6
assert mesh.n_nodes == int(np.prod(np.asarray(labels.shape) + 1))
for phase, count in expected_counts.items():
    region_cells = int(np.count_nonzero(mesh.cell_regions == phase))
    assert region_cells == count * 6, (phase, region_cells, count * 6)

# 5.5 FEM 几何质量
quality = result.fem.quality
assert quality.fem_ready
assert quality.inverted_cells == 0
assert quality.degenerate_cells == 0
assert quality.minimum_volume > 0.0
assert quality.minimum_mean_ratio > 0.0

# 5.6 物理坐标范围：数组为 z-y-x，节点为 x-y-z
spacing_zyx = np.asarray(condition.voxel_size_m, dtype=float)
expected_xyz_max = np.asarray(
    [labels.shape[2] * spacing_zyx[2], labels.shape[1] * spacing_zyx[1], labels.shape[0] * spacing_zyx[0]]
)
assert np.allclose(mesh.nodes.min(axis=0), np.zeros(3), rtol=0.0, atol=1e-18)
assert np.allclose(mesh.nodes.max(axis=0), expected_xyz_max, rtol=1e-12, atol=1e-18)

# 5.7 所有产物存在且非空
assert all(path.is_file() and path.stat().st_size > 0 for path in result.artifacts.values())
for required in (
    "volume-npz", "volume-npy", "volume-raw", "volume-metadata",
    "mesh-vtk", "mesh-msh", "mesh-inp", "mesh-boundary-ply", "mesh-boundary-stl",
    "mesh-quality", "condition", "generation", "metrics", "manifest",
):
    assert required in result.artifacts, required

# 5.8 NPZ / NPY / RAW 往返一致性
restored_npz = MicrostructureVolume.load_npz(result.artifacts["volume-npz"])
restored_npy = np.load(result.artifacts["volume-npy"], allow_pickle=False)
restored_raw = np.fromfile(result.artifacts["volume-raw"], dtype="<i4").reshape(labels.shape)
assert np.array_equal(restored_npz.labels, labels)
assert restored_npz.voxel_size_m == condition.voxel_size_m
assert np.array_equal(restored_npy, labels)
assert np.array_equal(restored_raw, labels)

# 5.9 TIFF 往返一致性（安装 Pillow 时）
if "volume-tiff" in result.artifacts:
    assert Image is not None and ImageSequence is not None
    with Image.open(result.artifacts["volume-tiff"]) as image:
        restored_tiff = np.stack(
            [np.asarray(frame.copy(), dtype=np.int32) for frame in ImageSequence.Iterator(image)],
            axis=0,
        )
    assert np.array_equal(restored_tiff, labels)

# 5.10 Manifest 完整性
manifest = json.loads(result.manifest.read_text(encoding="utf-8"))
assert manifest["status"] == "completed"
assert manifest["workflow"] == "zynnova.zynmorph.generate"
event_names = [event["name"] for event in manifest["events"]]
for required_event in (
    "generation_started", "generation_completed", "meshing_started", "meshing_completed"
):
    assert required_event in event_names

print("✅ 所有硬门禁均已通过。")


## 6. 三个正交切片与相组成可视化


In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

phases = list(condition.phases)
phase_to_index = {phase: index for index, phase in enumerate(phases)}
indexed_labels = np.empty_like(labels, dtype=np.int32)
for phase, index in phase_to_index.items():
    indexed_labels[labels == phase] = index

base_cmap = plt.get_cmap("tab10")
phase_colors = [base_cmap(index % 10) for index in range(len(phases))]
cmap = ListedColormap(phase_colors)
norm = BoundaryNorm(np.arange(-0.5, len(phases) + 0.5, 1), cmap.N)

z0, y0, x0 = (size // 2 for size in labels.shape)
views = [
    (indexed_labels[z0, :, :], f"z={z0}", "x", "y"),
    (indexed_labels[:, y0, :], f"y={y0}", "x", "z"),
    (indexed_labels[:, :, x0], f"x={x0}", "y", "z"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
for ax, (image, title, xlabel, ylabel) in zip(axes, views, strict=True):
    shown = ax.imshow(image, origin="lower", interpolation="nearest", cmap=cmap, norm=norm)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

cbar = fig.colorbar(shown, ax=axes, ticks=np.arange(len(phases)), shrink=0.86)
cbar.ax.set_yticklabels([BatteryPhase(phase).name for phase in phases])
fig.suptitle(f"{ELECTRODE_KIND.capitalize()} electrode — orthogonal label slices")
plt.show()

fraction_plot = phase_metrics_df.set_index("phase")[["target_fraction", "observed_fraction"]]
fraction_plot.plot(kind="bar", figsize=(9, 4), rot=15)
plt.ylabel("Volume fraction")
plt.title("Target and achieved phase fractions")
plt.tight_layout()
plt.show()


## 7. 轻量三维界面体素视图

这里显示发生相变化的体素以及外表面体素，用于快速检查拓扑。它不是替代有限元网格查看器的精确表面渲染。


In [ ]:
def phase_boundary_mask(values: np.ndarray) -> np.ndarray:
    mask = np.zeros(values.shape, dtype=bool)
    for axis in range(3):
        left_sel = [slice(None)] * 3
        right_sel = [slice(None)] * 3
        left_sel[axis] = slice(0, -1)
        right_sel[axis] = slice(1, None)
        changed = values[tuple(left_sel)] != values[tuple(right_sel)]
        left_mask = [slice(None)] * 3
        right_mask = [slice(None)] * 3
        left_mask[axis] = slice(0, -1)
        right_mask[axis] = slice(1, None)
        mask[tuple(left_mask)] |= changed
        mask[tuple(right_mask)] |= changed
    mask[0, :, :] = True
    mask[-1, :, :] = True
    mask[:, 0, :] = True
    mask[:, -1, :] = True
    mask[:, :, 0] = True
    mask[:, :, -1] = True
    return mask

boundary = phase_boundary_mask(labels)
coordinates_zyx = np.argwhere(boundary)
max_points = 18_000
if len(coordinates_zyx) > max_points:
    rng = np.random.default_rng(SEED)
    coordinates_zyx = coordinates_zyx[rng.choice(len(coordinates_zyx), max_points, replace=False)]

spacing = np.asarray(condition.voxel_size_m)
coordinates_xyz_um = coordinates_zyx[:, [2, 1, 0]] * spacing[[2, 1, 0]] * 1e6
point_phases = labels[tuple(coordinates_zyx.T)]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
for index, phase in enumerate(phases):
    keep = point_phases == phase
    ax.scatter(
        coordinates_xyz_um[keep, 0],
        coordinates_xyz_um[keep, 1],
        coordinates_xyz_um[keep, 2],
        s=5,
        alpha=0.55,
        label=BatteryPhase(phase).name,
        color=phase_colors[index],
    )
ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm)")
ax.set_zlabel("z (µm)")
ax.set_box_aspect(expected_xyz_max)
ax.set_title("Phase-boundary voxel view")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()


## 8. 界面面积、连通性与有限元网格质量


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

phase_metrics_df.set_index("phase")["connected_fraction"].plot(kind="bar", ax=axes[0], rot=15)
axes[0].set_ylim(0.0, 1.05)
axes[0].set_ylabel("Largest connected component / phase voxels")
axes[0].set_title("Connected fraction")

phase_metrics_df.set_index("phase")["specific_surface_1_per_um"].plot(kind="bar", ax=axes[1], rot=15)
axes[1].set_ylabel("Specific surface area (1/µm)")
axes[1].set_title("Phase specific surface area")
plt.show()

signed_volumes = tetrahedron_signed_volumes(mesh)
mean_ratios = tetrahedron_mean_ratio(mesh)

mesh_summary_df = pd.DataFrame(
    [
        {"metric": "nodes", "value": mesh.n_nodes},
        {"metric": "Tet4 cells", "value": mesh.n_cells},
        {"metric": "external boundary triangles", "value": result.fem.boundary.n_faces},
        {"metric": "material interface pairs", "value": len(result.fem.interface_faces)},
        {"metric": "minimum volume (m³)", "value": quality.minimum_volume},
        {"metric": "median volume (m³)", "value": quality.median_volume},
        {"metric": "maximum volume (m³)", "value": quality.maximum_volume},
        {"metric": "inverted cells", "value": quality.inverted_cells},
        {"metric": "degenerate cells", "value": quality.degenerate_cells},
        {"metric": "minimum mean ratio", "value": quality.minimum_mean_ratio},
        {"metric": "median mean ratio", "value": quality.median_mean_ratio},
        {"metric": "FEM ready", "value": quality.fem_ready},
    ]
)
display(mesh_summary_df)

interface_face_df = pd.DataFrame(
    [
        {
            "phase_pair": str(pair),
            "phase_names": f"{BatteryPhase(pair[0]).name} / {BatteryPhase(pair[1]).name}",
            "triangles": int(np.asarray(faces).shape[0]),
        }
        for pair, faces in sorted(result.fem.interface_faces.items())
    ]
)
display(interface_face_df)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(mean_ratios, bins=40)
ax.set_xlabel("Tet4 mean-ratio quality")
ax.set_ylabel("Cell count")
ax.set_title("Finite-element cell quality distribution")
plt.tight_layout()
plt.show()

assert np.all(signed_volumes > 0.0)
assert np.all(mean_ratios > 0.0)


## 9. 导出产物清单

- `microstructure.vtk`：通用非结构网格；
- `microstructure.msh`：Gmsh 2.2 Tet4，材料相写入物理标签；
- `microstructure.inp`：Abaqus `C3D4`，每个相写入独立 `ELSET`；
- `boundary.ply` / `boundary.stl`：外边界表面；
- `microstructure.npz`：保留体素尺寸、原点和相名称的首选体素格式；
- `manifest.json`：运行配置、事件和产物哈希。


In [ ]:
artifact_df = pd.DataFrame(
    [
        {
            "role": role,
            "suffix": path.suffix,
            "size_MiB": path.stat().st_size / 2**20,
            "path": str(path),
        }
        for role, path in sorted(result.artifacts.items())
    ]
).sort_values(["role"])
display(artifact_df)


## 10. 随机种子可复现性

这一节只调用生成器，不重复创建有限元网格。


In [ ]:
generator = SpectralConditionalGenerator()
repeat_a = generator.generate(condition, refinement_steps=0, temperature=config.temperature)
repeat_b = generator.generate(condition, refinement_steps=0, temperature=config.temperature)
assert np.array_equal(repeat_a.volume.labels, repeat_b.volume.labels)
assert dict(repeat_a.achieved_counts) == expected_counts

other_condition = replace(
    condition,
    seed=condition.seed + 1,
    descriptor_targets={},
    percolation_axes={},
)
other = generator.generate(other_condition, refinement_steps=0, temperature=config.temperature)
different_seed_agreement = float(np.mean(repeat_a.volume.labels == other.volume.labels))
assert different_seed_agreement < 1.0

print("同一条件、同一种子：逐体素完全一致。")
print(f"改变种子后的逐体素一致率：{different_seed_agreement:.4f}")


## 11. 相关长度条件扫描（可选）

固定相分数和随机种子，只改变活性材料的相关长度，检查形貌统计是否随条件变化。此处不生成网格，以便快速比较。


In [ ]:
sweep_df = pd.DataFrame()

if RUN_PARAMETER_SWEEP:
    sweep_lengths = [1.6, 3.2, 5.2]
    sweep_records = []
    sweep_images = []
    sweep_shape = (12, 24, 24) if FAST_MODE else (24, 40, 40)

    for active_length in sweep_lengths:
        lengths = dict(condition.correlation_lengths_voxels)
        lengths[ACTIVE_PHASE] = (active_length, active_length, active_length)
        sweep_condition = replace(
            condition,
            shape=sweep_shape,
            correlation_lengths_voxels=lengths,
            percolation_axes={},
            descriptor_targets={},
        )
        generated = generator.generate(sweep_condition, refinement_steps=0, temperature=config.temperature)
        metrics = analyze_microstructure(generated.volume)
        active_metric = metrics.phases[ACTIVE_PHASE]
        electrolyte_metric = metrics.phases[ELECTROLYTE_PHASE]
        sweep_records.append(
            {
                "active_correlation_length_vox": active_length,
                "active_connected_fraction": active_metric.connected_fraction,
                "active_specific_surface_1_per_um": active_metric.specific_surface_area_per_m / 1e6,
                "electrolyte_connected_fraction": electrolyte_metric.connected_fraction,
                "active_electrolyte_interface_um2": metrics.interface_area_m2.get(
                    tuple(sorted((ACTIVE_PHASE, ELECTROLYTE_PHASE))), 0.0
                ) * 1e12,
            }
        )
        sweep_images.append(generated.volume.labels[sweep_shape[0] // 2])
        assert dict(generated.achieved_counts) == dict(sweep_condition.exact_phase_counts())

    sweep_df = pd.DataFrame(sweep_records)
    display(sweep_df)

    fig, axes = plt.subplots(1, len(sweep_images), figsize=(15, 4), constrained_layout=True)
    for ax, image, length in zip(axes, sweep_images, sweep_lengths, strict=True):
        indexed = np.empty_like(image, dtype=np.int32)
        for phase, index in phase_to_index.items():
            indexed[image == phase] = index
        ax.imshow(indexed, origin="lower", interpolation="nearest", cmap=cmap, norm=norm)
        ax.set_title(f"active L={length:.1f} vox")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
    plt.show()

    sweep_df.plot(
        x="active_correlation_length_vox",
        y=["active_connected_fraction", "electrolyte_connected_fraction"],
        marker="o",
        figsize=(8, 4.5),
    )
    plt.ylim(0.0, 1.05)
    plt.ylabel("Connected fraction")
    plt.title("Connectivity response to correlation length")
    plt.tight_layout()
    plt.show()
else:
    print("RUN_PARAMETER_SWEEP=False，跳过条件扫描。")


## 12. 从二维标签切片重建三维体素场（可选）

该功能是“多平面投票 + 谱先验”的确定性重建基线，不应被误认为已经训练好的 CT/FIB-SEM 扩散模型。测试会：

- 从原三维结果抽取两个不同方向的低分辨率标签切片；
- 用新的随机先验重建三维结构；
- 检查观测平面是否被严格写回；
- 报告全体素一致率与重建后的形貌指标。


In [ ]:
reconstruction_metrics_df = pd.DataFrame()
reconstruction_agreement = None


def nearest_resize_2d(values: np.ndarray, shape: tuple[int, int]) -> np.ndarray:
    indices = [
        np.rint(np.linspace(0, values.shape[axis] - 1, shape[axis])).astype(np.int64)
        for axis in range(2)
    ]
    return values[np.ix_(indices[0], indices[1])]


if RUN_SLICE_RECONSTRUCTION:
    z_fraction = 0.35
    x_fraction = 0.65
    z_index = int(round(z_fraction * (labels.shape[0] - 1)))
    x_index = int(round(x_fraction * (labels.shape[2] - 1)))

    observed_z_lowres = labels[z_index, ::2, ::2]
    observed_x_lowres = labels[::2, ::2, x_index]
    observations = [
        SliceObservation(observed_z_lowres, axis=0, index_fraction=z_fraction, weight=1.8),
        SliceObservation(observed_x_lowres, axis=2, index_fraction=x_fraction, weight=1.4),
    ]

    reconstruction_condition = replace(
        condition,
        seed=condition.seed + 101,
        percolation_axes={},
        descriptor_targets={},
    )
    reconstructed = reconstruct_from_slices(
        observations,
        reconstruction_condition,
        prior_weight=0.35,
    )
    reconstructed_labels = reconstructed.labels

    expected_z = nearest_resize_2d(observed_z_lowres, (labels.shape[1], labels.shape[2]))
    expected_x = nearest_resize_2d(observed_x_lowres, (labels.shape[0], labels.shape[1]))
    # 观测按输入顺序写回；两个平面的交线由后写入的 x 平面决定。
    assert np.array_equal(reconstructed_labels[:, :, x_index], expected_x)
    z_plane_mask = np.ones(expected_z.shape, dtype=bool)
    z_plane_mask[:, x_index] = False
    assert np.array_equal(
        reconstructed_labels[z_index, :, :][z_plane_mask],
        expected_z[z_plane_mask],
    )
    assert np.array_equal(
        reconstructed_labels[z_index, :, x_index],
        expected_x[z_index, :],
    )

    reconstruction_agreement = float(np.mean(reconstructed_labels == labels))
    reconstruction_metrics = analyze_microstructure(reconstructed)
    reconstruction_metrics_df = pd.DataFrame(
        [
            {
                "phase": BatteryPhase(phase).name,
                "volume_fraction": metric.volume_fraction,
                "connected_fraction": metric.connected_fraction,
                "percolates_z": metric.percolates[0],
                "percolates_y": metric.percolates[1],
                "percolates_x": metric.percolates[2],
            }
            for phase, metric in reconstruction_metrics.phases.items()
        ]
    )
    print(f"原结构与重建结构的逐体素一致率：{reconstruction_agreement:.4f}")
    display(reconstruction_metrics_df)

    original_z_indexed = np.empty_like(labels[z_index], dtype=np.int32)
    recon_z_indexed = np.empty_like(reconstructed_labels[z_index], dtype=np.int32)
    original_x_indexed = np.empty_like(labels[:, :, x_index], dtype=np.int32)
    recon_x_indexed = np.empty_like(reconstructed_labels[:, :, x_index], dtype=np.int32)
    for phase, index in phase_to_index.items():
        original_z_indexed[labels[z_index] == phase] = index
        recon_z_indexed[reconstructed_labels[z_index] == phase] = index
        original_x_indexed[labels[:, :, x_index] == phase] = index
        recon_x_indexed[reconstructed_labels[:, :, x_index] == phase] = index

    recon_views = [
        (original_z_indexed, "Original observed z-plane"),
        (recon_z_indexed, "Reconstructed z-plane"),
        (original_x_indexed, "Original observed x-plane"),
        (recon_x_indexed, "Reconstructed x-plane"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
    for ax, (image, title) in zip(axes.ravel(), recon_views, strict=True):
        ax.imshow(image, origin="lower", interpolation="nearest", cmap=cmap, norm=norm)
        ax.set_title(title)
    plt.show()
else:
    print("RUN_SLICE_RECONSTRUCTION=False，跳过二维切片重建。")


## 13. 训练型 Rectified-Flow 后端入口

内置注册表还包含 `torch-rectified-flow`，但它必须接收与你的数据和条件键一致的检查点。没有检查点时，后端应明确报告不可用，而不是静默退回或伪造推理结果。


In [ ]:
flow_status = backend_df.loc[backend_df["name"] == "torch-rectified-flow"].iloc[0].to_dict()
display(pd.DataFrame([flow_status]))

FLOW_CHECKPOINT: str | None = None  # 例如 "/path/to/zynmorph_flow.pt"
if FLOW_CHECKPOINT is None:
    print("未配置 FLOW_CHECKPOINT：按预期跳过训练型后端推理。")
else:
    flow_config = replace(
        config,
        backend="torch-rectified-flow",
        output_directory=str(OUTPUT_ROOT / "torch_flow"),
    )
    flow_result = run_zynmorph(
        condition,
        flow_config,
        backend_options={"checkpoint": FLOW_CHECKPOINT, "device": "auto"},
    )
    print("Flow run:", flow_result.directory)


## 14. 生成最终机器可读测试报告


In [ ]:
test_summary = {
    "schema": "zynnova.zynmorph.notebook-test.v1",
    "status": "passed",
    "zynnova_version": zynnova.__version__,
    "electrode_kind": ELECTRODE_KIND,
    "backend": result.generation.backend,
    "shape_zyx": list(labels.shape),
    "voxel_size_m_zyx": list(condition.voxel_size_m),
    "physical_size_m_zyx": list(result.generation.volume.physical_size_m),
    "elapsed_seconds": float(elapsed_seconds),
    "run_directory": str(result.directory),
    "exact_counts": {str(k): int(v) for k, v in expected_counts.items()},
    "phase_metrics": {
        str(phase): {
            "name": BatteryPhase(phase).name,
            "volume_fraction": float(result.metrics.phases[phase].volume_fraction),
            "connected_fraction": float(result.metrics.phases[phase].connected_fraction),
            "percolates_zyx": [bool(v) for v in result.metrics.phases[phase].percolates],
            "specific_surface_area_per_m": float(result.metrics.phases[phase].specific_surface_area_per_m),
        }
        for phase in condition.phases
    },
    "mesh": {
        "nodes": int(mesh.n_nodes),
        "tetrahedra": int(mesh.n_cells),
        "boundary_triangles": int(result.fem.boundary.n_faces),
        "inverted_cells": int(quality.inverted_cells),
        "degenerate_cells": int(quality.degenerate_cells),
        "minimum_mean_ratio": float(quality.minimum_mean_ratio),
        "median_mean_ratio": float(quality.median_mean_ratio),
        "fem_ready": bool(quality.fem_ready),
    },
    "reproducibility": {
        "same_seed_exact": True,
        "different_seed_voxel_agreement": float(different_seed_agreement),
    },
    "slice_reconstruction": {
        "executed": bool(RUN_SLICE_RECONSTRUCTION),
        "voxel_agreement": None if reconstruction_agreement is None else float(reconstruction_agreement),
    },
    "parameter_sweep": {
        "executed": bool(RUN_PARAMETER_SWEEP),
        "rows": sweep_df.to_dict(orient="records") if not sweep_df.empty else [],
    },
    "artifacts": {role: str(path) for role, path in result.artifacts.items()},
}

summary_path = result.directory / "notebook_test_summary.json"
summary_path.write_text(
    json.dumps(test_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("✅ Notebook 测试报告：", summary_path)
display(pd.DataFrame([{
    "status": test_summary["status"],
    "run_directory": test_summary["run_directory"],
    "elapsed_seconds": test_summary["elapsed_seconds"],
    "voxels": labels.size,
    "tetrahedra": mesh.n_cells,
    "fem_ready": quality.fem_ready,
}]))


## 结论判据

当第 5 节输出：

```text
✅ 所有硬门禁均已通过。
```

并且最终报告中的 `status` 为 `passed`、`mesh.fem_ready` 为 `true` 时，说明当前 ZynMorph 的本地谱生成与结构化 Tet4 路径已经通过本 notebook 所覆盖的功能测试。

这不等于训练型扩散/Rectified-Flow 后端已经验证。训练型后端只有在提供真实检查点并运行第 13 节后，才可以对其质量和条件响应作结论。
